# 2.0 County-Level EDA

Exploratory data analysis for county-level outage statistics.

Includes:
- Frequency statistics for yearly outages, customer_hours, customer_hours_per_capita
- Seasonality analysis (by month)
- Time of day analysis (by hour)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json

sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load county-level data
data_dir = Path("../../data/processed/introduction")

with open(data_dir / "county_year_summary.json") as f:
    county_summary = pd.DataFrame(json.load(f))

with open(data_dir / "county_year_by_month.json") as f:
    county_month = pd.DataFrame(json.load(f))

with open(data_dir / "county_year_by_hour.json") as f:
    county_hour = pd.DataFrame(json.load(f))

print(f"County summary records: {len(county_summary):,}")
print(f"County by month records: {len(county_month):,}")
print(f"County by hour records: {len(county_hour):,}")

In [ ]:
county_summary.head()

In [ ]:
county_summary.info()

## Yearly Statistics

Summary statistics for outage counts, customer hours, and customer hours per capita.

In [ ]:
# Basic descriptive statistics
county_summary[['outage_count', 'customer_hours', 'customer_hours_per_capita']].describe()

In [ ]:
# Annual aggregates across all counties
yearly_totals = county_summary.groupby('year').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'fips': 'nunique'
}).rename(columns={'fips': 'counties_with_outages'})

print("Yearly totals across all counties:")
yearly_totals

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Outage count by year
axes[0].bar(yearly_totals.index, yearly_totals['outage_count'], color='steelblue')
axes[0].set_title('Total Outages by Year')
axes[0].set_xlabel('Year')
axes[0].set_ylabel('Outage Count')
axes[0].tick_params(axis='x', rotation=45)

# Customer hours by year
axes[1].bar(yearly_totals.index, yearly_totals['customer_hours'] / 1e6, color='coral')
axes[1].set_title('Total Customer Hours by Year')
axes[1].set_xlabel('Year')
axes[1].set_ylabel('Customer Hours (millions)')
axes[1].tick_params(axis='x', rotation=45)

# Counties with outages by year
axes[2].bar(yearly_totals.index, yearly_totals['counties_with_outages'], color='teal')
axes[2].set_title('Counties with Outages by Year')
axes[2].set_xlabel('Year')
axes[2].set_ylabel('Number of Counties')
axes[2].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

### Distribution of County-Level Metrics

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# Outage count distribution
axes[0, 0].hist(county_summary['outage_count'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].set_title('Distribution of Outage Count')
axes[0, 0].set_xlabel('Outage Count')
axes[0, 0].set_ylabel('Frequency')

axes[1, 0].hist(county_summary['outage_count'], bins=50, edgecolor='black', alpha=0.7)
axes[1, 0].set_title('Distribution of Outage Count (log scale)')
axes[1, 0].set_xlabel('Outage Count')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].set_yscale('log')

# Customer hours distribution
axes[0, 1].hist(county_summary['customer_hours'], bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[0, 1].set_title('Distribution of Customer Hours')
axes[0, 1].set_xlabel('Customer Hours')
axes[0, 1].set_ylabel('Frequency')

axes[1, 1].hist(np.log10(county_summary['customer_hours'] + 1), bins=50, edgecolor='black', alpha=0.7, color='coral')
axes[1, 1].set_title('Distribution of Customer Hours (log10)')
axes[1, 1].set_xlabel('log10(Customer Hours)')
axes[1, 1].set_ylabel('Frequency')

# Customer hours per capita distribution
axes[0, 2].hist(county_summary['customer_hours_per_capita'], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[0, 2].set_title('Distribution of Customer Hours per Capita')
axes[0, 2].set_xlabel('Customer Hours per Capita')
axes[0, 2].set_ylabel('Frequency')

axes[1, 2].hist(np.log10(county_summary['customer_hours_per_capita'] + 0.001), bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[1, 2].set_title('Distribution of Customer Hours per Capita (log10)')
axes[1, 2].set_xlabel('log10(Customer Hours per Capita)')
axes[1, 2].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

In [ ]:
# Percentile analysis
percentiles = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99]
metrics = ['outage_count', 'customer_hours', 'customer_hours_per_capita']

percentile_df = county_summary[metrics].quantile(percentiles)
percentile_df.index = [f"{int(p*100)}%" for p in percentiles]
print("Percentile distribution:")
percentile_df

### Top Counties by Various Metrics

In [ ]:
# Top 10 counties by total outages (across all years)
county_totals = county_summary.groupby(['fips', 'state', 'county']).agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
}).reset_index()

print("Top 10 counties by total outage count:")
county_totals.nlargest(10, 'outage_count')[['state', 'county', 'outage_count', 'customer_hours']]

In [ ]:
print("Top 10 counties by total customer hours:")
county_totals.nlargest(10, 'customer_hours')[['state', 'county', 'outage_count', 'customer_hours']]

In [ ]:
# Top 10 counties by average customer hours per capita
county_avg_per_capita = county_summary.groupby(['fips', 'state', 'county'])['customer_hours_per_capita'].mean().reset_index()

print("Top 10 counties by average customer hours per capita:")
county_avg_per_capita.nlargest(10, 'customer_hours_per_capita')

## Seasonality Analysis (Monthly)

Analyze outage patterns by month of year.

In [ ]:
county_month.head()

In [ ]:
# Aggregate by month across all counties and years
monthly_totals = county_month.groupby('month').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
}).reset_index()

month_names = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 
               'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
monthly_totals['month_name'] = monthly_totals['month'].apply(lambda x: month_names[x-1])

monthly_totals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Outage count by month
axes[0].bar(monthly_totals['month_name'], monthly_totals['outage_count'], color='steelblue')
axes[0].set_title('Total Outages by Month (All Years)')
axes[0].set_xlabel('Month')
axes[0].set_ylabel('Outage Count')

# Customer hours by month
axes[1].bar(monthly_totals['month_name'], monthly_totals['customer_hours'] / 1e6, color='coral')
axes[1].set_title('Total Customer Hours by Month (All Years)')
axes[1].set_xlabel('Month')
axes[1].set_ylabel('Customer Hours (millions)')

plt.tight_layout()
plt.show()

In [ ]:
# Month by year heatmap for outage count
monthly_by_year = county_month.groupby(['year', 'month'])['outage_count'].sum().unstack()

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(monthly_by_year, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax)
ax.set_title('Outage Count by Year and Month')
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

In [ ]:
# Month by year heatmap for customer hours
customer_hours_by_year = county_month.groupby(['year', 'month'])['customer_hours'].sum().unstack() / 1e6

fig, ax = plt.subplots(figsize=(14, 6))
sns.heatmap(customer_hours_by_year, annot=True, fmt='.1f', cmap='YlOrRd', ax=ax)
ax.set_title('Customer Hours (millions) by Year and Month')
ax.set_xlabel('Month')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly statistics
monthly_stats = county_month.groupby('month').agg({
    'outage_count': ['sum', 'mean', 'std', 'median'],
    'customer_hours': ['sum', 'mean', 'std', 'median']
})
monthly_stats.columns = ['_'.join(col) for col in monthly_stats.columns]
monthly_stats.index = month_names

print("Monthly statistics:")
monthly_stats

## Time of Day Analysis (Hourly)

Analyze outage patterns by hour of day.

In [ ]:
county_hour.head()

In [ ]:
# Aggregate by hour across all counties and years
hourly_totals = county_hour.groupby('hour').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
}).reset_index()

hourly_totals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Outage count by hour
axes[0].bar(hourly_totals['hour'], hourly_totals['outage_count'], color='steelblue')
axes[0].set_title('Total Outages by Hour of Day (All Years)')
axes[0].set_xlabel('Hour (0-23)')
axes[0].set_ylabel('Outage Count')
axes[0].set_xticks(range(0, 24))

# Customer hours by hour
axes[1].bar(hourly_totals['hour'], hourly_totals['customer_hours'] / 1e6, color='coral')
axes[1].set_title('Total Customer Hours by Hour of Day (All Years)')
axes[1].set_xlabel('Hour (0-23)')
axes[1].set_ylabel('Customer Hours (millions)')
axes[1].set_xticks(range(0, 24))

plt.tight_layout()
plt.show()

In [ ]:
# Hour by year heatmap for outage count
hourly_by_year = county_hour.groupby(['year', 'hour'])['outage_count'].sum().unstack()

fig, ax = plt.subplots(figsize=(16, 6))
sns.heatmap(hourly_by_year, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax, annot_kws={'size': 8})
ax.set_title('Outage Count by Year and Hour')
ax.set_xlabel('Hour')
ax.set_ylabel('Year')
plt.tight_layout()
plt.show()

In [ ]:
# Define time of day periods
def categorize_hour(hour):
    if 6 <= hour < 12:
        return 'Morning (6-11)'
    elif 12 <= hour < 18:
        return 'Afternoon (12-17)'
    elif 18 <= hour < 22:
        return 'Evening (18-21)'
    else:
        return 'Night (22-5)'

county_hour['time_period'] = county_hour['hour'].apply(categorize_hour)

time_period_totals = county_hour.groupby('time_period').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum'
})

# Reorder for logical presentation
time_period_order = ['Morning (6-11)', 'Afternoon (12-17)', 'Evening (18-21)', 'Night (22-5)']
time_period_totals = time_period_totals.reindex(time_period_order)

print("Outages by time of day:")
time_period_totals

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart for outage count by time period
axes[0].pie(time_period_totals['outage_count'], labels=time_period_totals.index, 
            autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[0].set_title('Outage Distribution by Time of Day')

# Pie chart for customer hours by time period
axes[1].pie(time_period_totals['customer_hours'], labels=time_period_totals.index, 
            autopct='%1.1f%%', colors=sns.color_palette('Set2'))
axes[1].set_title('Customer Hours Distribution by Time of Day')

plt.tight_layout()
plt.show()

In [ ]:
# Hourly statistics
hourly_stats = county_hour.groupby('hour').agg({
    'outage_count': ['sum', 'mean', 'std', 'median'],
    'customer_hours': ['sum', 'mean', 'std', 'median']
})
hourly_stats.columns = ['_'.join(col) for col in hourly_stats.columns]

print("Hourly statistics:")
hourly_stats

## State-Level Summary

In [ ]:
# Aggregate by state
state_totals = county_summary.groupby('state').agg({
    'outage_count': 'sum',
    'customer_hours': 'sum',
    'fips': 'nunique'
}).rename(columns={'fips': 'counties_count'}).sort_values('customer_hours', ascending=False)

print("Top 15 states by customer hours:")
state_totals.head(15)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Top 20 states by outage count
top_states_outages = state_totals.nlargest(20, 'outage_count')
axes[0].barh(top_states_outages.index[::-1], top_states_outages['outage_count'][::-1], color='steelblue')
axes[0].set_title('Top 20 States by Outage Count')
axes[0].set_xlabel('Outage Count')

# Top 20 states by customer hours
top_states_hours = state_totals.nlargest(20, 'customer_hours')
axes[1].barh(top_states_hours.index[::-1], top_states_hours['customer_hours'][::-1] / 1e6, color='coral')
axes[1].set_title('Top 20 States by Customer Hours')
axes[1].set_xlabel('Customer Hours (millions)')

plt.tight_layout()
plt.show()